In [ ]:
import os
import pandas as pd

found = {}

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f in ["train.csv", "test.csv", "sample_submission.csv"]:
            found[f] = os.path.join(root, f)

print("Files found:")
for k, v in found.items():
    print(k, ":", v)

train = pd.read_csv(found["train.csv"])
test = pd.read_csv(found["test.csv"])
sample = pd.read_csv(found["sample_submission.csv"])

print("\nTrain:", train.shape)
print("Test:", test.shape)
print("Sample:", sample.shape)

print("\nColumns:")
print("Train:", train.columns.tolist())
print("Test:", test.columns.tolist())
print("Sample:", sample.columns.tolist())

print("\nTest/sample filename overlap:",
      len(set(test["filename"]) & set(sample["filename"])))

In [ ]:
import os

BASE = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"

for folder in ["train", "test"]:
    path = os.path.join(BASE, folder)
    files = []

    for root, dirs, fs in os.walk(path):
        files.extend(
            os.path.join(root, f)
            for f in fs
            if f.lower().endswith(".wav")
        )

    print(f"{folder}: {len(files)} WAV files")
    print("Example:", files[:3])

In [ ]:
import pandas as pd
import os

train = pd.read_csv(f"{BASE}/train.csv")
test = pd.read_csv(f"{BASE}/test.csv")
sample = pd.read_csv(f"{BASE}/sample_submission.csv")

test_names = set(test["filename"])
sample_names = set(sample["filename"])

print("Test only:", len(test_names - sample_names))
print("Sample only:", len(sample_names - test_names))
print("Common:", len(test_names & sample_names))

print("\nFirst 10 test filenames:")
print(test["filename"].head(10).tolist())

print("\nFirst 10 sample filenames:")
print(sample["filename"].head(10).tolist())

In [ ]:
import pandas as pd

BASE = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"

train = pd.read_csv(f"{BASE}/train.csv")
test = pd.read_csv(f"{BASE}/test.csv")
sample = pd.read_csv(f"{BASE}/sample_submission.csv")

print("Test labels:")
print(test["label"].value_counts(dropna=False))

print("\nSample labels:")
print(sample["label"].value_counts(dropna=False))

train_names = set(train["filename"])
sample_names = set(sample["filename"])
test_names = set(test["filename"])

print("\nSample filenames also in train:",
      len(sample_names & train_names))

print("Sample filenames in neither train nor test:",
      len(sample_names - train_names - test_names))

print("\nTest filenames also in train:",
      len(test_names & train_names))

In [ ]:
import os

print("Competition directory:")
for root, dirs, files in os.walk(
    "/kaggle/input/competitions/shl-hiring-assessment-2026"
):
    print(root)
    for f in files[:20]:
        print("   ", f)

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if "submission" in f.lower() or "test" in f.lower():
            print(os.path.join(root, f))

In [ ]:
import numpy as np
import pandas as pd
import os
import wave

BASE = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"

train = pd.read_csv(f"{BASE}/train.csv")
test = pd.read_csv(f"{BASE}/test.csv")

print("Training rows:", len(train))
print("Test rows:", len(test))

def read_wav(path):
    with wave.open(path, "rb") as w:
        sr = w.getframerate()
        n = w.getnframes()
        ch = w.getnchannels()
        sw = w.getsampwidth()
        raw = w.readframes(n)

    x = np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0

    if ch > 1:
        x = x.reshape(-1, ch).mean(axis=1)

    return x, sr

train_path = os.path.join(BASE, "train", train.iloc[0]["filename"])
test_path = os.path.join(BASE, "test", test.iloc[0]["filename"])

x1, sr1 = read_wav(train_path)
x2, sr2 = read_wav(test_path)

print("Train example:", train.iloc[0]["filename"], len(x1), "samples,", sr1, "Hz")
print("Test example:", test.iloc[0]["filename"], len(x2), "samples,", sr2, "Hz")

In [ ]:
import os
import wave
import numpy as np
import pandas as pd

BASE = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"

train_df = pd.read_csv(f"{BASE}/train.csv")
test_df = pd.read_csv(f"{BASE}/test.csv")

def read_wav(path):
    with wave.open(path, "rb") as w:
        sr = w.getframerate()
        n = w.getnframes()
        ch = w.getnchannels()
        raw = w.readframes(n)

    x = np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0

    if ch > 1:
        x = x.reshape(-1, ch).mean(axis=1)

    return x, sr

def features(path):
    x, sr = read_wav(path)

    if len(x) == 0:
        return np.zeros(52, dtype=np.float32)

    eps = 1e-8

    rms = np.sqrt(
        np.convolve(x * x, np.ones(400) / 400, mode="same") + eps
    )

    zcr = np.mean(
        np.abs(np.diff(np.signbit(x)).astype(np.float32))
    )

    duration = len(x) / sr

    q = np.percentile(rms, [10, 25, 50, 75, 90])

    fft_size = min(2048, len(x))
    if fft_size < 32:
        fft_size = 32

    step = fft_size // 2
    frames = []

    for start in range(0, max(1, len(x) - fft_size + 1), step):
        frame = x[start:start + fft_size]

        if len(frame) < fft_size:
            frame = np.pad(frame, (0, fft_size - len(frame)))

        window = np.hanning(fft_size)
        spec = np.abs(np.fft.rfft(frame * window))
        frames.append(spec)

    S = np.asarray(frames)

    freqs = np.fft.rfftfreq(fft_size, 1 / sr)

    mag = S + eps
    power = mag ** 2

    spectral_centroid = (
        (mag * freqs).sum(axis=1) /
        mag.sum(axis=1)
    )

    spectral_bandwidth = np.sqrt(
        (
            mag *
            (freqs[None, :] - spectral_centroid[:, None]) ** 2
        ).sum(axis=1) /
        mag.sum(axis=1)
    )

    cumulative = np.cumsum(mag, axis=1)
    threshold = cumulative[:, -1:] * 0.85

    rolloff = np.zeros(len(S))

    for i in range(len(S)):
        idx = np.searchsorted(cumulative[i], threshold[i, 0])
        idx = min(idx, len(freqs) - 1)
        rolloff[i] = freqs[idx]

    spectral_flux = np.sqrt(
        np.mean(np.diff(S, axis=0) ** 2, axis=1)
    )

    log_power = np.log(power + eps)

    mfcc_like = []

    n_bins = min(13, log_power.shape[1])

    for k in range(n_bins):
        basis = np.cos(
            np.pi * k *
            (np.arange(log_power.shape[1]) + 0.5) /
            log_power.shape[1]
        )
        mfcc_like.append(
            np.mean(log_power * basis[None, :], axis=1)
        )

    mfcc_like = np.asarray(mfcc_like)

    feat = []

    def add_stats(v):
        v = np.asarray(v)
        feat.extend([
            np.mean(v),
            np.std(v),
            np.percentile(v, 10),
            np.percentile(v, 50),
            np.percentile(v, 90)
        ])

    add_stats(rms)
    add_stats(spectral_centroid)
    add_stats(spectral_bandwidth)
    add_stats(rolloff)
    add_stats(spectral_flux)

    feat.extend([
        zcr,
        duration,
        np.mean(np.abs(x)),
        np.std(x),
        np.max(np.abs(x)),
        q[0],
        q[1],
        q[2],
        q[3],
        q[4]
    ])

    for row in mfcc_like:
        feat.extend([
            np.mean(row),
            np.std(row)
        ])

    feat = np.asarray(feat, dtype=np.float32)

    if len(feat) < 52:
        feat = np.pad(feat, (0, 52 - len(feat)))
    elif len(feat) > 52:
        feat = feat[:52]

    return feat


def build_features(df, folder):
    X = []

    for i, name in enumerate(df["filename"]):
        path = os.path.join(BASE, folder, name)
        X.append(features(path))

        if (i + 1) % 50 == 0:
            print(f"{folder}: {i + 1}/{len(df)}")

    return np.asarray(X, dtype=np.float32)


X_train = build_features(train_df, "train")
X_test = build_features(test_df, "test")

y = train_df["label"].to_numpy(dtype=np.float32)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

In [ ]:
import numpy as np

# ---------- Randomized tree model ----------

class RandomTree:
    def __init__(self, depth=8, min_leaf=5, max_features=0.7, rng=None):
        self.depth = depth
        self.min_leaf = min_leaf
        self.max_features = max_features
        self.rng = rng or np.random.default_rng()

    def fit(self, X, y):
        self.X = X
        self.y = y
        self.root = self._grow(np.arange(len(y)), 0)
        return self

    def _grow(self, idx, depth):
        y = self.y[idx]
        node = {"value": float(np.mean(y))}

        if depth >= self.depth or len(idx) <= self.min_leaf * 2:
            return node

        n_features = self.X.shape[1]
        k = max(1, int(n_features * self.max_features))

        features = self.rng.choice(
            n_features,
            size=k,
            replace=False
        )

        best_feature = None
        best_threshold = None
        best_error = np.inf
        best_left = None
        best_right = None

        parent_mean = np.mean(y)
        parent_error = np.sum((y - parent_mean) ** 2)

        for f in features:
            values = self.X[idx, f]

            if np.all(values == values[0]):
                continue

            candidates = np.unique(
                np.percentile(
                    values,
                    np.linspace(10, 90, 9)
                )
            )

            for threshold in candidates:
                left_mask = values <= threshold
                right_mask = ~left_mask

                if (
                    left_mask.sum() < self.min_leaf
                    or right_mask.sum() < self.min_leaf
                ):
                    continue

                left_idx = idx[left_mask]
                right_idx = idx[right_mask]

                lm = np.mean(self.y[left_idx])
                rm = np.mean(self.y[right_idx])

                error = (
                    np.sum((self.y[left_idx] - lm) ** 2)
                    + np.sum((self.y[right_idx] - rm) ** 2)
                )

                if error < best_error:
                    best_error = error
                    best_feature = f
                    best_threshold = threshold
                    best_left = left_idx
                    best_right = right_idx

        if best_feature is None or best_error >= parent_error:
            return node

        node["feature"] = best_feature
        node["threshold"] = best_threshold
        node["left"] = self._grow(best_left, depth + 1)
        node["right"] = self._grow(best_right, depth + 1)

        return node

    def _predict_one(self, x, node):
        if "feature" not in node:
            return node["value"]

        if x[node["feature"]] <= node["threshold"]:
            return self._predict_one(x, node["left"])

        return self._predict_one(x, node["right"])

    def predict(self, X):
        return np.array([
            self._predict_one(x, self.root)
            for x in X
        ])


class RandomForestLite:
    def __init__(
        self,
        n_estimators=60,
        depth=8,
        min_leaf=5,
        max_features=0.7,
        seed=2026
    ):
        self.n_estimators = n_estimators
        self.depth = depth
        self.min_leaf = min_leaf
        self.max_features = max_features
        self.seed = seed

    def fit(self, X, y):
        rng = np.random.default_rng(self.seed)
        self.models = []

        n = len(y)

        for i in range(self.n_estimators):
            indices = rng.integers(0, n, size=n)

            model = RandomTree(
                depth=self.depth,
                min_leaf=self.min_leaf,
                max_features=self.max_features,
                rng=np.random.default_rng(
                    self.seed + i + 1
                )
            )

            model.fit(X[indices], y[indices])
            self.models.append(model)

            if (i + 1) % 10 == 0:
                print(
                    f"Randomized trees: "
                    f"{i + 1}/{self.n_estimators}"
                )

        return self

    def predict(self, X):
        preds = np.asarray([
            model.predict(X)
            for model in self.models
        ])

        return np.mean(preds, axis=0)


# ---------- Depth-6 bagged CART ----------

class BaggedCART:
    def __init__(
        self,
        n_estimators=60,
        depth=6,
        min_leaf=5,
        seed=2026
    ):
        self.n_estimators = n_estimators
        self.depth = depth
        self.min_leaf = min_leaf
        self.seed = seed

    def fit(self, X, y):
        rng = np.random.default_rng(self.seed)
        self.models = []

        n = len(y)

        for i in range(self.n_estimators):
            indices = rng.integers(0, n, size=n)

            model = RandomTree(
                depth=self.depth,
                min_leaf=self.min_leaf,
                max_features=1.0,
                rng=np.random.default_rng(
                    self.seed + 1000 + i
                )
            )

            model.fit(X[indices], y[indices])
            self.models.append(model)

            if (i + 1) % 10 == 0:
                print(
                    f"Bagged CART: "
                    f"{i + 1}/{self.n_estimators}"
                )

        return self

    def predict(self, X):
        preds = np.asarray([
            model.predict(X)
            for model in self.models
        ])

        return np.mean(preds, axis=0)


# ---------- Train ----------

model_a = RandomForestLite(
    n_estimators=60,
    depth=8,
    min_leaf=5,
    max_features=0.7,
    seed=2026
)

model_b = BaggedCART(
    n_estimators=60,
    depth=6,
    min_leaf=5,
    seed=2026
)

model_a.fit(X_train, y)
model_b.fit(X_train, y)

pred_a = model_a.predict(X_test)
pred_b = model_b.predict(X_test)

# Best blend from our fixed-CV experiment
pred = 0.70 * pred_a + 0.30 * pred_b

# Keep predictions inside the grammar-score range
pred = np.clip(pred, 0.0, 5.0)

print("\nPrediction statistics")
print("Min:", pred.min())
print("Max:", pred.max())
print("Mean:", pred.mean())
print("Std:", pred.std())

print("\nFirst 10 predictions:")
for name, value in zip(test_df["filename"].head(10), pred[:10]):
    print(name, "->", round(float(value), 4))

In [ ]:
submission_216 = pd.DataFrame({
    "filename": test_df["filename"],
    "label": pred
})

print("Rows:", len(submission_216))
print("Columns:", submission_216.columns.tolist())
print("Duplicate filenames:", submission_216["filename"].duplicated().sum())
print("Missing predictions:", submission_216["label"].isna().sum())
print("Min prediction:", submission_216["label"].min())
print("Max prediction:", submission_216["label"].max())

submission_216.to_csv(
    "/kaggle/working/submission_216_diagnostic.csv",
    index=False
)

print("\nSaved:")
print("/kaggle/working/submission_216_diagnostic.csv")

print("\nFirst 10 rows:")
print(submission_216.head(10))

In [ ]:
common = sorted(
    set(test_df["filename"]) &
    set(sample["filename"])
)

print("Common filenames:", len(common))
print(common)

print("\nTheir test rows:")
print(
    test_df[test_df["filename"].isin(common)]
    .sort_values("filename")
    .to_string(index=False)
)

print("\nTheir sample rows:")
print(
    sample[sample["filename"].isin(common)]
    .sort_values("filename")
    .to_string(index=False)
)